### Trade-off: Monte Carlo vs TD Learning

The question is:
>how we estimate future returns using experience?

<br>
Comparison:

| | Monte Carlo (MC) | Temporal Difference (TD) |
|---|---|---|
| **Core idea** | Learn from actual sampled returns | Learn from reward + estimated future value |
| **Learning target** | $G_t$ | $r_{t+1} + \gamma V(s_{t+1})$ |
| **When to update** | After complete return is available | After each step (TD(0)) |
| **Bootstrapping** | No | Yes |
| **Variance** | Often higher | Often lower |
| **Bias** | Sampled return is unbiased for the true value | Target may be biased due to inaccurate $V$ |
| **Main advantage** | Learns from observed outcomes without relying on future value estimates | Can learn before an episode finishes |
| **Main limitation** | Must wait for future rewards | Relies on potentially inaccurate estimates |
| **Best suited for** | Episodic tasks with manageable episode lengths | Long episodes or continuing tasks |



<br><br>
Bellman Expectation Equation (skip derivation):

$$
V^\pi(s)
=
\mathbb E_\pi[
r_{t+1}+\gamma V^\pi(s_{t+1})
\mid s_t=s
]
$$

It describes the math relationship for value function. But not how to learn it.

**If we know P distribution, we can solve V.**

But
>without knowing the environment's probability distributions, can we find the solution to the Bellman Equation?

Yes! TD is basically how to use sampling to approximate the solution. 

TD(0), a.k.a 1 step, equation:

$$
V(s)\leftarrow
V(s)+\alpha[
r+\gamma V(s')-V(s)
]
$$


### Bias–Variance Trade-off Intuition

1 state: 

50% -> 0

50% -> +10

So the expected return is 5. Let's say TD estimate is 4.


In [1]:
future = [0, 10]
true_V = 5

mc = future
td = [4, 4]  # Estimated V(next_state) = 4

for name, xs in [("MC", mc), ("TD", td)]:
    mean = sum(xs) / len(xs)
    bias = mean - true_V
    variance = sum((x - mean)**2 for x in xs) / len(xs)
    print(name, "bias:", bias, "variance:", variance)

MC bias: 0.0 variance: 25.0
TD bias: -1.0 variance: 0.0


- MC: high variance (unstable, since it's sampled!), but when sampled a lot the average is true.

- TD: low variance (stable, since it's estimated!) but coule be wrong (bias).

Ideally, if TD estimate is Really good, there is low variance and bias. You can have your cake and eat it!

Key point:
> variance and bias are independant. Same wrong guess repeats 100 times has 0 variance.


In [1]:
def update_q(s, a, reward, next_s, terminated):
    # Do not bootstrap from a terminal state.
    target = reward if terminated else reward + GAMMA * Q[next_s].max()
    Q[s, a] += ALPHA * (target - Q[s, a])

### Reward propagation

We have:

$$
S_0\rightarrow S_1\rightarrow S_2\rightarrow S_3\rightarrow R=1
$$

All states initial value is 0.

once MC get a full smaple, it can use return = 1 to update all states.

After observing one complete episode, MC knows the actual return from every state:

\begin{aligned}
V(S_3)&=1\\
V(S_2)&=0.9\\
V(S_1)&=0.81\\
V(S_0)&=0.729
\end{aligned}


TD(0) updates each state using its immediate reward and the current estimate of the next state's value:

$$
V(S_t)\leftarrow V(S_t)+\alpha\big[R_{t+1}+\gamma V(S_{t+1})-V(S_t)\big]
$$

First, TD(0) visits $S_0,S_1,S_2$, but values are still 0. Only $S_3$ receives an update.

The reward information propagates backward over multiple episodes:

| After episode | V(S₀) | V(S₁) | V(S₂) | V(S₃) |
| ------------- | ----- | ----- | ----- | ----- |
| Initial       | 0     | 0     | 0     | 0     |
| 1             | 0     | 0     | 0     | 1     |
| 2             | 0     | 0     | 0.9   | 1     |
| 3             | 0     | 0.81  | 0.9   | 1     |
| 4             | 0.729 | 0.81  | 0.9   | 1     |

ex. in episode 2, the agent is at S2, then -> act -> go to S3, then we use S3's value as a (estimated) reward for S2. 

### Credit assignment

This is related to "value estimation" but it's a **different question**.

TD/MC focus on:
> how to estimate return.

Credit assignment focus on:
> now we get the return. How do we distribute credit? to which action, or whether action doesn't matter it's just the state or policy, etc.